# cross-judge quickstart: 3-model ensemble judge

This notebook demonstrates the cross-judge v0.1 public API end-to-end:

1. Build 3 mock critics (Claude / DeepSeek / Kimi) that return canned responses (no live LLM calls).
2. Wire them into an `Ensemble` with `voting='majority'`.
3. Judge a query and inspect the consensus + agreement + Krippendorff α.
4. Swap voting strategies, observe how consensus changes.
5. Replace the mock clients with real OpenAI-compatible vendors (commented out — needs API keys).

**Why a mock notebook?** Live LLM calls cost money + need API keys + are non-deterministic. The structure is identical to live usage; the only difference is we inject a `http_client` mock instead of letting Critic build a real httpx client.

## Cell 1 — Install (skip if you ran `pip install -e packages/cross-judge`)

In [ ]:
# %pip install cross-judge --quiet

## Cell 2 — Imports + mock HTTP client (no live calls)

In [ ]:
import json
from dataclasses import dataclass, field
from typing import Any

from cross_judge import Critic, Ensemble, Verdict, VerdictKind
from cross_judge.voting import krippendorff_alpha, agreement_pct

print('cross-judge imports OK')

## Cell 3 — Mock client (stands in for httpx.Client)

In [ ]:
@dataclass
class MockResponse:
    body: dict
    def raise_for_status(self): pass
    def json(self): return self.body

@dataclass
class MockHTTPXClient:
    content: str  # the LLM's response content
    calls: list = field(default_factory=list)
    def post(self, url, json, headers):
        self.calls.append({'url': url, 'json': json})
        return MockResponse(body={'choices': [{'message': {'content': self.content}}]})

def mock_critic(name, model, vendor, kind, conf=0.9, reasoning='ok', temperature=0.0):
    body = json.dumps({'kind': kind, 'confidence': conf, 'reasoning': reasoning})
    return Critic(
        name=name,
        model=model,
        vendor=vendor,
        temperature=temperature,
        api_key='mock-key',
        http_client=MockHTTPXClient(content=body),
    )

print('mock client ready')

## Cell 4 — Build 3 critics: Claude (strict KEEP) / DeepSeek (creative KEEP) / Kimi (REJECT)

In [ ]:
claude = mock_critic('claude-strict',   'anthropic/claude-sonnet-4.5', 'openrouter', kind='KEEP',   conf=0.92, reasoning='Power-law tails match.')
deepseek = mock_critic('ds-pro-creative','deepseek-v4-pro',              'deepseek',   kind='KEEP',   conf=0.85, reasoning='Same scaling exponent (≈1.5).', temperature=0.7)
kimi = mock_critic('kimi-rigor',         'moonshot/kimi-k2',             'openrouter', kind='REJECT', conf=0.75, reasoning='Mechanism differs — bank failures = contagion, quakes = stress release.')

critics = [claude, deepseek, kimi]
print(f'{len(critics)} critics ready: {[c.name for c in critics]}')

## Cell 5 — Wire into an Ensemble + judge a query

In [ ]:
ensemble = Ensemble(critics=critics, voting='majority')

result = ensemble.judge(
    query='Are bank failure cascades and earthquake aftershocks in the same universality class?',
    query_id='cand-001',
)

print(f'Consensus:        {result.consensus}')
print(f'Agreement:        {result.agreement_pct:.0%}')
print(f'Avg confidence:   {result.avg_confidence:.2f}')
print(f'Krippendorff α:   {result.krippendorff_alpha:.3f}')
print(f'Disagreement?     {result.disagreement}')
print()
for v in result.verdicts:
    print(f'  {v.critic_id:20s} kind={v.kind:8s} conf={v.confidence:.2f}  {v.reasoning}')

## Cell 6 — Krippendorff α interpretation

With 2 KEEP + 1 REJECT, α ≈ 0.0: the agreement is at chance level given the marginal distribution. Compare to:

- **3 KEEP, 0 REJECT** → α = 1.0 (perfect agreement)
- **2 KEEP, 1 REJECT** → α ≈ 0.0 (chance-level)
- **1 KEEP, 1 REJECT** → α = 0.0 (no information)

Common acceptance thresholds in content analysis:
- α > 0.8 → ship it
- 0.667 ≤ α ≤ 0.8 → substantial agreement
- α < 0.4 → surface to human review

In [ ]:
# Verify the three regimes:

def make_synthetic_verdicts(labels):
    return [Verdict(critic_id=f'c{i}', kind=k, confidence=0.9, reasoning='') for i, k in enumerate(labels)]

for labels in [['KEEP', 'KEEP', 'KEEP'], ['KEEP', 'KEEP', 'REJECT'], ['KEEP', 'REJECT'], ['KEEP', 'REJECT', 'SPLIT']]:
    a = krippendorff_alpha(make_synthetic_verdicts(labels))
    print(f'  labels={labels!s:42s}  α = {a if a is None else round(a, 3)}')

## Cell 7 — Swap voting strategies

In [ ]:
ensemble_unanimous = Ensemble(
    critics=critics,
    voting='unanimous',
    voting_kwargs={'fallback': 'NEEDS_HUMAN_REVIEW'},
)
r2 = ensemble_unanimous.judge('q', query_id='cand-001-unanimous')
print(f'Unanimous consensus: {r2.consensus}  (disagree={r2.disagreement})')

## Cell 8 — Tie-break via priority list

In [ ]:
# 2 critics, opposite verdicts: priority=['REJECT', 'KEEP'] → REJECT wins ties
two = [mock_critic('a', 'm', 'deepseek', 'KEEP'), mock_critic('b', 'm', 'deepseek', 'REJECT')]
tie = Ensemble(two, voting='majority', voting_kwargs={'priority': ['REJECT', 'KEEP']})
r3 = tie.judge('q')
print(f'Tied 1-1, priority=[REJECT, KEEP] → {r3.consensus}')

## Cell 9 — Real vendor usage (commented out; needs API keys)

In [ ]:
# import os
# os.environ.setdefault('OPENROUTER_API_KEY', 'sk-or-...')
# os.environ.setdefault('DEEPSEEK_API_KEY',  'sk-...')
#
# live_critics = [
#     Critic(name='claude-strict',   model='anthropic/claude-sonnet-4.5', vendor='openrouter', temperature=0.0),
#     Critic(name='ds-pro-creative', model='deepseek-v4-pro',             vendor='deepseek',   temperature=0.7),
#     Critic(name='kimi-rigor',      model='moonshot/kimi-k2',            vendor='openrouter', temperature=0.0),
# ]
# live_ensemble = Ensemble(critics=live_critics, voting='majority')
# live_result = live_ensemble.judge('Your real query here.')
# print(live_result.consensus, live_result.krippendorff_alpha)

## Cell 10 — Summary

You've seen:

- `Critic(name, model, vendor, prompt_template, ...)` — one LLM critic.
- `Ensemble(critics, voting=...)` — N critics voting on the same query.
- `Verdict(kind, confidence, reasoning, critic_id, ...)` — single verdict.
- `EnsembleVerdict(consensus, agreement_pct, krippendorff_alpha, ...)` — aggregate.
- Mock httpx client injection for offline testing.
- Voting strategies: `majority` (default), `unanimous`, custom callable.
- Krippendorff α interpretation.

Next steps: load the bundled YAML prompts via `Critic.from_yaml_prompt(...)`,
or write your own under `prompts/your-judge@v0.1.yaml` and git-tag for
reproducible methods sections.